# CropForecastLK: Phase 2 — Data Cleaning & Feature Engineering
**Sri Lanka Highland Crops Production Forecasting & Agricultural Intelligence System**

---

### Phase 2 Overview
In this notebook, we transform the raw multi-sheet census dataset into a high-quality, feature-engineered tabular dataset ready for predictive machine learning models:
- **Step 5**: Regex string cleaning parser (`clean_numeric_string`), stripping commas and placeholders (`"-"`, `"n.a."`).
- **Step 6**: Agronomic anomaly detection (handling zero-extent records) and cohort-based group-median imputation.
- **Step 7**: Domain yield ratio derivation ($Crop\_Yield = Production / Extent$), 1-year temporal lags, and 3-year rolling extent statistics.
- **Step 8**: Out-of-fold smoothed target encodings for categorical features (District, Crop) and binary Season encoding.
- **Step 9**: Strict chronological temporal train/validation/test splitting (Train: 2000–2017, Val: 2018–2020, Test: 2021–2023) to completely prevent data leakage.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

# Add ml_pipeline to sys.path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from ml_pipeline.config import RAW_DATA_PATH, CLEANED_DATA_PATH, ENGINEERED_FEATURES_PATH
from ml_pipeline.preprocessing import preprocess_raw_data, clean_numeric_string, filter_aggregate_rows
from ml_pipeline.feature_engineering import run_feature_engineering, compute_yield_ratios, create_temporal_features


### Step 5: Regex String Cleaning & Aggregate Filtering
We execute the preprocessing routines to strip commas, drop `"National Total"` and aggregate `"Total"` seasons, and standardize strings.

In [ ]:
# Run raw preprocessing
df_cleaned = preprocess_raw_data(raw_path=RAW_DATA_PATH, save_path=CLEANED_DATA_PATH)

print(f"Cleaned dataset rows: {len(df_cleaned):,}")
print(f"Remaining null Extent: {df_cleaned['Extent'].isna().sum()}")
print(f"Remaining null Production: {df_cleaned['Production'].isna().sum()}")
display(df_cleaned.head(6))


### Step 6: Distribution of Extent and Production Post-Cleaning
Verify that numeric conversions and group-median imputations produced smooth, non-null continuous distributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(np.log1p(df_cleaned['Extent']), kde=True, ax=axes[0], color='#2a9d8f', bins=35)
axes[0].set_title("Log-Transformed Extent (Hectares) Distribution", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Log(1 + Extent)")

sns.histplot(np.log1p(df_cleaned['Production']), kde=True, ax=axes[1], color='#e76f51', bins=35)
axes[1].set_title("Log-Transformed Production (Metric Tons) Distribution", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Log(1 + Production)")

plt.tight_layout()
plt.show()


### Steps 7 - 9: Yield Ratios, Temporal Lags & Chronological Splitting
We compute the domain yield ratios ($MT/Ha$), 1-year lagged production and yield, 3-year rolling extent statistics, target encodings, and temporal splits.

In [ ]:
# Run complete feature engineering pipeline
train_df, val_df, test_df, encoder = run_feature_engineering(
    cleaned_path=CLEANED_DATA_PATH,
    output_path=ENGINEERED_FEATURES_PATH
)

print(f"Training Set (2000 - 2017): {len(train_df):,} records")
print(f"Validation Set (2018 - 2020): {len(val_df):,} records")
print(f"Test Set (2021 - 2023): {len(test_df):,} records")


In [ ]:
# Display newly engineered temporal and domain features
feature_cols = [
    "District", "Crop", "Season", "Year", "Extent", "Production",
    "Crop_Yield", "Production_Lag_1Y", "Yield_Lag_1Y",
    "Extent_RollMean_3Y", "District_TargetEnc", "Crop_TargetEnc", "Season_Maha"
]
display(train_df[feature_cols].dropna().head(10))


### Temporal Verification of Chronological Splits
Confirm that the train, validation, and test partitions have zero overlap across the temporal axis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

split_data = [
    ("Training (2000-2017)", len(train_df), '#264653'),
    ("Validation (2018-2020)", len(val_df), '#2a9d8f'),
    ("Holdout Test (2021-2023)", len(test_df), '#e76f51')
]

bars = ax.bar([s[0] for s in split_data], [s[1] for s in split_data], color=[s[2] for s in split_data], width=0.45)
ax.set_title("Chronological Data Partitioning (Zero Future-to-Past Leakage)", fontsize=13, fontweight='bold')
ax.set_ylabel("Number of Agricultural Records")

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 500, f"{int(yval):,}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


### Summary of Phase 2 Deliverables
1. **Cleaned CSV**: Exported to `data/processed/cleaned_highland_crops.csv`.
2. **Engineered Features Parquet/CSV**: Exported to `data/processed/engineered_features.csv`.
3. **Temporal Discipline**: Zero data leakage guaranteed through chronological splitting and training-fold target encoding.
4. **Next Step**: Proceed to `03_benchmarking_crossval_evaluation.ipynb` to evaluate 5 candidate regression models via Time-Series Cross Validation.
